# 02 — Lire les résultats et leurs limites

Exécuter d’abord `python -m src.demo`. Ce notebook lit par défaut ses sorties synthétiques. Modifier les chemins ensemble pour lire une expérience réelle. Il ne lance pas d’entraînement et ne sélectionne pas un modèle sur le test.

In [ ]:
from pathlib import Path
import os
import sys
import json
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
from src.visualization.report import load_evaluation

RUN = ROOT / "outputs/demo_synthetic/run"
EVALUATION = ROOT / "outputs/demo_synthetic/evaluation"
REPORT = ROOT / "outputs/demo_synthetic/report"
EXPLANATIONS = ROOT / "outputs/demo_synthetic/gradcam"
frame, metrics, provenance = load_evaluation(EVALUATION)
synthetic = provenance["dataset_kind"] == "synthetic"
display(Markdown("**SYNTHÉTIQUE : les scores ne mesurent aucune performance médicale.**" if synthetic else "**Lire les effectifs, le protocole et les annotations avec les scores.**"))
provenance

## 1. Apprentissage et sélection sur validation

La perte d’entraînement mesure l’objectif optimisé. L’AP de validation sert à sélectionner le checkpoint. Leurs échelles sont différentes ; les tracer séparément. Une époque de démonstration ne donne pas une tendance d’apprentissage.

In [ ]:
history = pd.read_csv(RUN / "history.csv")
display(history[["epoch", "train_loss", "average_precision"]])
axes = history.plot(x="epoch", y=["train_loss", "average_precision"], subplots=True, marker="o", figsize=(8, 5), title=("SYNTHÉTIQUE — " if synthetic else "") + "Historique")

## 2. Erreurs au seuil enregistré

Sensibilité = TP/(TP+FN), spécificité = TN/(TN+FP), précision = TP/(TP+FP). Une valeur non définie reste non définie. L’accuracy ne remplace pas l’analyse des faux négatifs.

In [ ]:
display(pd.Series(metrics["overall"], name="valeur").to_frame())
display(Image(filename=str(REPORT / "confusion_matrix.png")))
display(Image(filename=str(REPORT / "roc_pr.png")))

## 3. Généralisation et unités de comparaison

Les métriques par centre et par lame agrègent des prédictions de patchs. Ce ne sont pas des diagnostics de lame/patiente. Un seul centre dans le test interdit d’affirmer une comparaison entre plusieurs centres de test.

In [ ]:
display(pd.DataFrame(metrics["by_center"]).T)
display(pd.DataFrame(metrics["by_slide"]).T)
print("Centres dans cette évaluation :", frame.center_id.nunique())
print("Groupes patient :", frame.patient_id.nunique())

## 4. Expliquer aussi les erreurs

La règle de sélection est écrite dans le CSV et l’index. Toutes les cartes expliquent la classe 1, même pour une prédiction négative. Une carte n’est ni une segmentation ni une preuve clinique.

In [ ]:
selection = pd.read_csv(EXPLANATIONS / "selection.csv")
display(selection[["case", "label", "score", "figure", "flat_heatmap"]])
for filename in selection.figure.head(4):
    display(Image(filename=str(EXPLANATIONS / filename)))

## 5. Questions avant de conclure

1. Le seuil et le modèle ont-ils été fixés avant l’évaluation du test ?
2. Quels faux négatifs observons-nous réellement, avec quels dénominateurs ?
3. Quel changement entre centres est mesuré, et quelles autres différences peuvent l’expliquer ?
4. Les résultats sont-ils stables ou proviennent-ils d’une seule graine ?
5. Les activations correspondent-elles aux annotations disponibles ?
6. Quelles informations manquent pour passer au ganglion/pN ?

Consigner : observation → interprétation → limite. Pour la démonstration, conclure seulement sur le bon fonctionnement du logiciel.